## Utility and imports

In [ ]:
from schema import common_mice
from schema.mpanze_paw_tracking_refactor import mpanze_paw_tracking_refactor as pt
from schema.mpanze_exp_refactor import mpanze_exp_refactor as exp
from schema.mpanze_widefield_refactor import mpanze_widefield_refactor as wf
import numpy as np
from pathlib import Path
import pandas as pd
from datetime import datetime
from tqdm.autonotebook import tqdm
import warnings
import matplotlib.pyplot as plt
from matplotlib import rc
import seaborn as sns
%matplotlib inline
import cv2

import rpy2.robjects as robjects
from rpy2.robjects import pandas2ri
from rpy2.robjects.packages import importr
from statannotations.Annotator import Annotator, PValueFormat

from scipy.ndimage import gaussian_filter1d

base = importr('base')
lme4 = importr('lme4')
emmeans = importr('emmeans')
stats = importr('stats')

# define path for datasets
p_datasets = Path('~/neurophys_3/r_outputs/datasets/').expanduser()
p_datasets.mkdir(parents=True, exist_ok=True)
p_figures = Path('~/neurophys_3/r_outputs/figures/figure_1/').expanduser()
p_figures.mkdir(parents=True, exist_ok=True)

Figure definitions - Manuscript

In [ ]:
# set font to Arial
rc('font',**{'family':'sans-serif','sans-serif':['Arial']})
# set font sizes to 12 for figures
rc('font', size=12)          # controls default text sizes
rc('axes', titlesize=12)     # fontsize of the axes title
rc('axes', labelsize=12)    # fontsize of the x and y labels
rc('xtick', labelsize=12)    # fontsize of the tick labels
rc('ytick', labelsize=12)    # fontsize of the tick labels
rc('legend', fontsize=10)    # legend fontsize
rc('figure', titlesize=12)  # fontsize of the figure title

# set line width to 1
rc('lines', linewidth=1)

# set dpi to 600 for figures
rc('figure', dpi=600)

# svg font type shenanigans
rc('svg', fonttype='none')

fontsize_small = 10
fontsize_medium = 12
fontsize_large = 14

# define conversion factor from inches to cm for convenience
cm = 1/2.54 * 1.5 # (scale larger for Manuscript)

# color palette for cohorts
group_colors = {'Sham':'#BBBBBB', 'Stroke':'#4477AA', 'Stroke + training':'#AA3377'}

## Figure 1B - Example data

In [ ]:
session = dict(
    mouse_id = 64,
    days_from_stroke_norm = -2,
    wf_param_id=6
)
# fetch widefield stack
key = (exp.DaysFromStrokeNorm * wf.ImageProcessingParameters2 & session).fetch1("KEY")
u, svt, h, w = (wf.ImageProcessing2 & key).load_components(svt_baseline=True)
stack_wf = (u@svt).T.reshape(-1, h, w)
t_wf = (wf.Synchronisation & key).fetch1("frame_timestamps_blue")
M = (wf.RescaledAllenRegistration2 & key).fetch1("allen_matrix_rescaled")
handedness = (exp.Handedness & key).fetch1("handedness")

# get task events
t_rew = (exp.JoystickExperiment.Trials & dict(**key, successful=1)).fetch("t_servo_out")
t_cue = (exp.JoystickExperiment.Trials & dict(**key)).fetch("t_cue")

t_0 = 141.1

# create figure
f, ax = plt.subplots(2,2, figsize=(4.5*cm, 5.5*cm), 
                     gridspec_kw={'width_ratios':[0.7,1], 'hspace':0, 'wspace':0, 'top':1, 'bottom':0, 'left':0, 'right':1})
for a in ax.flatten():
    a.set_axis_off()


# plot example df/f map
from mpanze_scripts.util.allen_utils import overlay_allen, load_allen
masks, area_names, edges, mask_total, bregma = load_allen((128, 128))
mask_combined = np.zeros((128, 128), dtype=np.uint8)
areas = [
    "MOs", "MOp", "SSp-ll", "SSp-ul", "SSp-nosemouth", "SSp-bfd", "SSp-tr",
    "RSP", "VISp", "VIS-medial", "VISa", "VISrl",
]
areas_to_overlay = [a+"_R" for a in areas] + [a+"_L" for a in areas]
areas_to_dot = ["MOs-medial_R","RSP-anterior_R"]

for i in range(len(area_names)):
    if area_names[i] in areas_to_overlay + ["SSp-un_R", "SSp-un_L"]:
        mask_combined[masks[i]>0] = 255
# fill holes
mask_combined = cv2.morphologyEx(mask_combined, cv2.MORPH_CLOSE, np.ones((5,5), np.uint8))
frame_0 = np.searchsorted(t_wf, t_0)
example_wf_frame = stack_wf[frame_0] * 100
example_wf_frame = cv2.warpAffine(example_wf_frame, M, (w,h))
if handedness == "R":
    example_wf_frame = np.fliplr(example_wf_frame)
example_wf_frame[mask_combined==0] = np.nan


imsh=ax[0,0].imshow(example_wf_frame, cmap='viridis', vmin=-0.5, vmax=1)
cax = ax[0,0].inset_axes([0.25,-0.1,0.5,0.15])
cbar = f.colorbar(imsh, cax=cax, orientation='horizontal')
cbar.ax.text(0.5, -0.2, '$\Delta F/F_0$ (%)', fontsize=fontsize_small, ha='center', va='top', transform=cbar.ax.transAxes)
# set ticks to sides left and right
cbar.ax.set_xticks([])
cbar.ax.text(-0.05, 0.5, '-0.5', fontsize=fontsize_small, ha='right', va='center', transform=cbar.ax.transAxes)
cbar.ax.text(1.05, 0.5, '1.0', fontsize=fontsize_small, ha='left', va='center', transform=cbar.ax.transAxes)
cbar.ax.tick_params(labelsize=fontsize_small)
overlay_allen(ax[0,0], res=(128,128), areas_to_overlay=areas_to_overlay, show_bregma=False,
              line_kw=dict(color='white', linewidth=0.5, alpha=.5))
overlay_allen(ax[0,0], res=(128,128), areas_to_overlay=["MOp_R"], line_kw=dict(color='magenta', linewidth=0.7), show_bregma=False)
overlay_allen(ax[0,0], res=(128,128), areas_to_overlay=["MOp_L"], line_kw=dict(color='orange', linewidth=0.7), show_bregma=False)
overlay_allen(ax[0,0], res=(128,128), areas_to_overlay=["MOs_R"], line_kw=dict(color='cyan', linewidth=0.7), show_bregma=False)
ax[0,0].text(0.47, 1.1, "Calcium imaging", fontsize=8, ha='center', va='top', transform=ax[0,0].transAxes)

# plot mop response ipsi-vs-contra
dff_1 = (wf.AllenSegmentation2.ROI & dict(**key, roi_id='MOp_contra')).fetch1("dff")
dff_2  = (wf.AllenSegmentation2.ROI & dict(**key, roi_id='MOp_ipsi')).fetch1("dff")
dff_3 = (wf.AllenSegmentation2.ROI & dict(**key, roi_id='MOs_contra')).fetch1("dff")
ax[0,1].plot(t_wf, dff_1*100, color='magenta', linewidth=0.75)
ax[0,1].plot(t_wf, dff_2*100, color='orange', linewidth=0.75)
ax[0,1].plot(t_wf, dff_3*100, color='cyan', linewidth=0.75)
ax[0,1].set_xlim(t_0-3, t_0+3)
ax[0,1].set_ylim(-1, 2)
ax[0,1].axvspan(t_0+2, t_0+3, ymin=0.05, ymax=0.07, color='k')
ax[0,1].axhspan(0.3,1.3, xmin=0.96, xmax=0.98, color='k')
ax[0,1].text(t_0+2.5, -1.3, '1 s', fontsize=fontsize_small, ha='center', va='center')
ax[0,1].text(t_0+1.7, 0.55, '1 %', fontsize=fontsize_small)
# ax[0,1].axvline(t_0, color='k', linestyle='--', linewidth=1, alpha=0.5)
for t_r in t_rew:
    ax[0,1].axvline(t_r, color='r', linestyle='--', linewidth=0.75, alpha=1)
for t_r in t_cue:
    ax[0,1].axvline(t_r, color='r', linestyle='--', linewidth=0.75, alpha=1)

# plot example paw frames
key_ipsi = (pt.PawRecording.Hand & dict(**key, side='ipsi')).fetch1("KEY")
t_ipsi = (pt.Synchronisation.Hand & key_ipsi).fetch1("frame_timestamps")
frame_ipsi = np.searchsorted(t_ipsi, t_0)
p_video = (pt.PawRecording.Hand & key_ipsi).get_path(as_posix=True)
cap = cv2.VideoCapture(p_video)
cap.set(cv2.CAP_PROP_POS_FRAMES, frame_ipsi)
ret, frame = cap.read()
cap.release()
frame_1 = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
# crop frame_1
crop_w = 600
crop_h = 500
crop_x = 250
crop_y = 100
frame_1 = frame_1[crop_y:crop_y+crop_h, crop_x:crop_x+crop_w]
# ax[1,0].imshow(frame)
key_contra = (pt.PawRecording.Hand & dict(**key, side='contra')).fetch1("KEY")
t_contra = (pt.Synchronisation.Hand & key_contra).fetch1("frame_timestamps")
frame_contra = np.searchsorted(t_contra, t_0)
p_video = (pt.PawRecording.Hand & key_contra).get_path(as_posix=True)
cap = cv2.VideoCapture(p_video)
cap.set(cv2.CAP_PROP_POS_FRAMES, frame_contra)
ret, frame = cap.read()
cap.release()
frame_2 = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
frame_2 = np.fliplr(frame_2)  # flip to match wf
# crop frame_2
frame_2 = frame_2[crop_y:crop_y+crop_h, crop_x:crop_x+crop_w]
# ax[2,0].imshow(np.fliplr(frame), cmap='gray', vmin=0, vmax=170)
# ax[2,0].imshow(np.fliplr(frame))
ax[1,0].imshow(np.vstack([frame_1, frame_2]), vmin=0, vmax=240, cmap='gray', aspect='equal')
ax[1,0].text(0, 0.5, "Task limb", fontsize=fontsize_medium, ha='left', va='bottom', transform=ax[1,0].transAxes, color='tab:blue')
ax[1,0].text(0, 0, "Support limb", fontsize=fontsize_small, ha='left', va='bottom', transform=ax[1,0].transAxes, color='tab:orange')

# add paw velocity
v_ipsi = (pt.WeightedHandPosition.Hand & key_ipsi).fetch_velocity_1(norm=True)
ax[1,1].plot(t_ipsi, v_ipsi, color='tab:blue', linewidth=0.75)
v_contra = (pt.WeightedHandPosition.Hand & key_contra).fetch_velocity_1(norm=True)
ax[1,1].plot(t_contra, v_contra, color='tab:orange', linewidth=0.75)
ax[1,1].set_ylim(-95, 95)
ax[1,1].set_xlim(t_0-3, t_0+3)
ax[1,1].axhspan(25,75, xmin=0.96, xmax=0.98, color='k')
ax[1,1].text(t_0+2.91, 85, 'limb velocity', fontsize=fontsize_small, ha='right', va='center')
ax[1,1].text(t_0+2.8, 50, '50 a.u.', fontsize=fontsize_small, ha='right', va='center')

# add rotation
rot_ipsi = (pt.Features.Feature & dict(**key_ipsi, label='rotation_24')).fetch1("feature")
rot_contra = (pt.Features.Feature & dict(**key_contra, label='rotation_24')).fetch1("feature")
ax_1_1_twin = ax[1,1].twinx()
ax_1_1_twin.plot(t_ipsi, rot_ipsi, color='tab:blue', linewidth=0.75)
ax_1_1_twin.plot(t_contra, rot_contra+20, color='tab:orange', linewidth=0.75)
ax_1_1_twin.set_ylim(0, 250)
ax_1_1_twin.axhspan(35,85, xmin=0.96, xmax=0.98, color='k')
ax_1_1_twin.text(t_0+2.91, 105, 'limb rotation', fontsize=fontsize_small, ha='right', va='center')
ax_1_1_twin.text(t_0+2.73, 60, '50°', fontsize=fontsize_small, ha='right', va='center')
ax_1_1_twin.set_axis_off()

plt.show()
f.savefig(p_figures / 'figure_1_example_data.svg', dpi=600, transparent=True)

## Figure 1D - lesion size

In [ ]:
df_stroke_volume = (
    exp.StrokeVolume()
    * exp.StrokeGroup.proj(stroke_group='group')
    & ["stroke_group='Stroke'", "stroke_group='Rehab'"]
).fetch(format='frame')

# save as csv for submission
df_stroke_volume.to_csv(p_figures / 'F1D_lesion_size_data.csv')

display(df_stroke_volume.head())
df_stroke_volume["stroke_group"] = df_stroke_volume["stroke_group"].replace({'Rehab':'Stroke + training'})
vol_rehab = df_stroke_volume.query("stroke_group=='Stroke + training'")['stroke_volume']
vol_stroke = df_stroke_volume.query("stroke_group=='Stroke'")['stroke_volume']
print(len(vol_rehab), len(vol_stroke))
from scipy.stats import ttest_ind
t, p = ttest_ind(vol_rehab, vol_stroke)

# print mean and sem for each group
mean_rehab = np.mean(vol_rehab)
sem_rehab = np.std(vol_rehab) / np.sqrt(len(vol_rehab))
mean_stroke = np.mean(vol_stroke)
sem_stroke = np.std(vol_stroke) / np.sqrt(len(vol_stroke))
print(f"Rehab: mean={mean_rehab:.2f}, sem={sem_rehab:.2f}")
print(f"Stroke: mean={mean_stroke:.2f}, sem={sem_stroke:.2f}")

# from scipy.stats import ranksums
# t, p = ranksums(vol_rehab, vol_stroke)
print(f"t={t:.2f}, p={p:.3f}")
f, ax = plt.subplots(1,1, figsize=(2*cm, 2*cm), 
                     gridspec_kw={'top':0.93, 'bottom':0.1, 'left':0.65, 'right':1})
sns.boxplot(
    data=df_stroke_volume.reset_index(),
    ax=ax,
    hue='stroke_group',
    y='stroke_volume',
    palette=group_colors,
    legend=False,
    fill=False,
    # dodge=True,
    gap=0.1,
    # width=0.5,
    whis=(0,100)
)
sns.stripplot(
    ax=ax,
    data=df_stroke_volume.reset_index(),
    hue='stroke_group',
    y='stroke_volume',
    palette=group_colors,
    dodge=True,
    # alpha=1,
    size=3,
    legend=False,
    alpha=0.7,
)
# turn off top, right
# ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
# ax.spines['bottom'].set_visible(False)
ax.set_xticks([])
ax.set_yticks([0,2,4])
ax.set_ylim(0, 5)
# ax.set_title('Lesion volume (mm$^3$)')
ax.set_ylabel('Lesion vol.\n(mm$^3$)')
ax.text(0, 4.5, f'p={p:.2f}', fontsize=fontsize_small, ha='center', va='bottom')
# plt.tight_layout()
ax.plot([-0.2,0.2], [4.5,4.5], color='k', linewidth=1)
plt.show()
f.savefig(p_figures / 'figure_1_lesion_size.svg', dpi=600, transparent=True)

## Datasets

### grasp onset-offset delimited - task limb

In [ ]:
keys = (
    pt.MovementSegmentation.Hand.proj("side")
    * exp.DaysFromStrokeNorm
    * exp.ExperimentalPhase
    * exp.StrokeGroup.proj(stroke_group='group')
    & "phase != 'Learning'"
    & "stroke_group != 'Learning'"
    & [f"days_from_stroke_norm = {d}" for d in [-3, -2, -1, 3, 7, 14, 21, 28]]
    & "mouse_id > 37"
    & "side = 'ipsi'"
).fetch("KEY")
print(f"Found {len(keys)} sessions")

labels_to_include = ["rotation_24", "open_alt_24", "distance", "velocity", "bend_24"]
df_onset_offset_ipsi = (
    (
        pt.MovementSegmentationSleap.Hand
        & keys
    )
    .epoch_features(subtract_baseline=False, labels_to_include=labels_to_include)
    # .reset_index()
    .join((exp.DaysFromStrokeNorm
           * pt.EpochClassification.Epoch.proj("epoch_class")
           & keys
           ).fetch(format='frame'))
    .reset_index()
    .query("epoch_class=='rewarded'")
    .drop(columns=["username", "day", "session_num", "pt_seg_id", "hand", "epoch_class"])
    .rename(columns={"days_from_stroke_norm":"day"})
    .set_index(["mouse_id", "day", "epoch_id"])
)

df_onset_offset_ipsi["rotation_24_range"] = df_onset_offset_ipsi["rotation_24_max"] - df_onset_offset_ipsi["rotation_24_min"]
df_onset_offset_ipsi["open_alt_24_range"] = df_onset_offset_ipsi["open_alt_24_max"] - df_onset_offset_ipsi["open_alt_24_min"]
df_onset_offset_ipsi["bend_24_range"] = df_onset_offset_ipsi["bend_24_max"] - df_onset_offset_ipsi["bend_24_min"]

df_onset_offset_ipsi.head()

# save dataset as pickle
df_onset_offset_ipsi.to_pickle(p_datasets / "df_onset_offset_ipsi.pkl")

### grasp onset-offset delimited - support limb

In [ ]:
keys = (
    pt.MovementSegmentation
    * exp.DaysFromStrokeNorm
    * exp.ExperimentalPhase
    * exp.StrokeGroup.proj(stroke_group='group')
    & "phase != 'Learning'"
    & "stroke_group != 'Learning'"
    & [f"days_from_stroke_norm = {d}" for d in [-3, -2, -1, 3, 7, 14, 21, 28]]
    & "mouse_id > 37"
).fetch("KEY")
print(f"Found {len(keys)} sessions")

labels_to_include = ["rotation_24", "open_alt_24", "distance", "velocity", "bend_24"]
rows = []
index = []

for key in tqdm(keys):
    d = (exp.DaysFromStrokeNorm & key).fetch1("days_from_stroke_norm")
    # get paw keys 
    key_ipsi = (pt.PawRecording.Hand & dict(**key, side='ipsi')).fetch1("KEY")
    key_contra = (pt.PawRecording.Hand & dict(**key, side='contra')).fetch1("KEY")
    
    # get features
    feature_matrix_contra, names = (pt.Features.Hand & key_contra).fetch_feature_matrix(labels_to_include=labels_to_include)

    # get ipsi paw onsets and offsets
    epoch_ids, epoch_starts, epoch_ends = (
        pt.MovementSegmentation.Epoch.proj("start_time", "end_time")
        * pt.EpochClassification.Epoch.proj("epoch_class")
        & dict(**key_ipsi, epoch_class='rewarded')
    ).fetch("epoch_id", "start_time", "end_time")

    # get contra paw synchronisation
    frame_timestamps_contra = (pt.Synchronisation.Hand & key_contra).fetch1("frame_timestamps")
    
    row = []
    # iterate over epochs
    for epoch_id, epoch_start, epoch_end in zip(epoch_ids, epoch_starts, epoch_ends):
        # find start and end frame
        start_frame_contra = np.searchsorted(frame_timestamps_contra, epoch_start)
        end_frame_contra = np.searchsorted(frame_timestamps_contra, epoch_end)
        
        data_contra = feature_matrix_contra[start_frame_contra:end_frame_contra]
        # compute features
        feat_mean = np.mean(data_contra, axis=0)
        feat_max = np.max(data_contra, axis=0)
        feat_min = np.min(data_contra, axis=0)
        feat_range = feat_max - feat_min
        rows.append(np.hstack([feat_mean, feat_max, feat_min, feat_range]))
        index.append([key["mouse_id"], d, epoch_id])

# create dataframe
index = pd.MultiIndex.from_tuples(index, names=["mouse_id", "day", 'epoch_id'])
df_onset_offset_contra = pd.DataFrame(rows, index=index, columns=[f"{n}_{stat}" for stat in ["mean", "max", "min", "range"] for n in names])
df_onset_offset_contra.to_pickle(p_datasets / "df_onset_offset_contra.pkl")
df_onset_offset_contra.head()


### Movement rates and performance

In [ ]:
keys = (
    pt.MovementSegmentation
    * exp.DaysFromStrokeNorm
    * exp.ExperimentalPhase
    * exp.StrokeGroup.proj(stroke_group='group')
    & "phase != 'Learning'"
    & "stroke_group != 'Learning'"
    & [f"days_from_stroke_norm = {d}" for d in [-3, -2, -1, 3, 7, 14, 21, 28]]
    & "mouse_id > 37"
).fetch("KEY")
print(f"Found {len(keys)} sessions")

# fetch movement rates
df_movements = (
    (
        pt.EpochClassification.Epoch
        * exp.DaysFromStrokeNorm
        * pt.PawRecording.Hand
        & "side='ipsi'"
        & keys
        & "epoch_class != 'excluded'"
    )
    .fetch(format='frame')
    .reset_index()
    .filter(["mouse_id", "days_from_stroke_norm", "epoch_id", "epoch_class"])
    .rename(columns={"days_from_stroke_norm": "day"})
    .groupby(["mouse_id", "day", "epoch_class"]).agg({"epoch_id": "count"})
    .rename(columns=dict(epoch_id='n_epochs'))
    .unstack(level="epoch_class", fill_value=0)
)
df_movements.columns = [c + "_rate" for c in df_movements.columns.get_level_values(1)]
df_movements.columns.name = None
df_time = (
    (
        exp.JoystickExperiment.Trials
        * exp.DaysFromStrokeNorm
        & keys
        & "trial_id=99"
    )
    .fetch(format='frame')
    .reset_index()
    .filter(["mouse_id", "days_from_stroke_norm", "t_end"])
    .rename(columns={"days_from_stroke_norm": "day"})
    .set_index(["mouse_id", "day"])
)

df_movements_contra = (
    (
        pt.MovementSegmentation.Epoch
        * exp.DaysFromStrokeNorm
        * pt.PawRecording.Hand
        & "side='contra'"
        & keys
    )
    .fetch(format='frame')
    .reset_index()
    .filter(["mouse_id", "days_from_stroke_norm", "epoch_id"])
    .rename(columns={"days_from_stroke_norm": "day"})
    .groupby(["mouse_id", "day"]).agg({"epoch_id": "count"})
    .rename(columns=dict(epoch_id='total_movement_rate_contra'))
)
df_movements = df_movements.join(df_movements_contra)

df_rates = df_movements.div(df_time.t_end, axis=0)
display(df_rates.head(10))
df_rates["total_movement_rate_ipsi"] = df_rates.sum(axis=1)
df_rates["accuracy"] = df_rates["rewarded_rate"] / (df_rates["rewarded_rate"] + df_rates["miss_rate"]) * 100

# fetch mouse performance
df_performance = (
    (
        exp.Performance
        * exp.DaysFromStrokeNorm
        * exp.StrokeGroup.proj(stroke_group='group')
        * exp.ExperimentalPhase
        & keys
    )
    .fetch(format='frame')
    .reset_index()
    .filter(["mouse_id", "days_from_stroke_norm", "performance", "phase", "stroke_group", "new_group"])
    .rename(columns={"days_from_stroke_norm": "day"})
    .set_index(["mouse_id", "day"])
)
df_performance["performance"] = df_performance["performance"] * 100
dataset_performance = df_performance.join(df_rates).drop(columns=["phase", "stroke_group"])

display(dataset_performance)
dataset_performance.to_pickle(p_datasets / "dataset_performance_rates.pkl")

### Experimental info

In [ ]:
keys = (
    pt.MovementSegmentation
    * exp.DaysFromStrokeNorm
    * exp.ExperimentalPhase
    * exp.StrokeGroup.proj(stroke_group='group')
    & "phase != 'Learning'"
    & "stroke_group != 'Learning'"
    & [f"days_from_stroke_norm = {d}" for d in [-3, -2, -1, 3, 7, 14, 21, 28]]
    & "mouse_id > 37"
).fetch("KEY")
print(f"Found {len(keys)} sessions")

dataset_info = (
    (
        exp.StrokeGroup.proj(stroke_group='group')
        * exp.ExperimentalPhase
        * exp.DaysFromStrokeNorm
        & keys
    )
    .fetch(format='frame')
    .reset_index()
    .filter(["mouse_id", "days_from_stroke_norm", "phase", "stroke_group"])
    .rename(columns={"days_from_stroke_norm": "day"})
    .set_index(["mouse_id", "day"])        
)
# rename groups
dataset_info["stroke_group"] = dataset_info["stroke_group"].replace({"Rehab": "Stroke + training"})
dataset_info["phase"] = dataset_info["phase"].replace({"Expert":"Pre", "Early":"Post Early", "Late":"Post Late"})
# create categorical variables - they work better with R for stats
dataset_info["p"] = pd.Categorical(dataset_info["phase"], categories=["Pre", "Post Early", "Post Late"], ordered=True)
dataset_info["g"] = pd.Categorical(dataset_info["stroke_group"], categories=["Sham", "Stroke", "Stroke + training"], ordered=True)
# day catergorical variable should be converted to string
dataset_info["d"] = pd.Categorical(dataset_info.index.get_level_values("day").astype(str), categories=[str(d) for d in [-3, -2, -1, 3, 7, 14, 21, 28]], ordered=True)
display(dataset_info)
dataset_info.to_pickle(p_datasets / "dataset_info.pkl")

### Supplementary figure 1C - support limb movements

In [ ]:
# plot changes in task performance
feature = "total_movement_rate_contra"
df_stats = dataset_performance.filter([feature]).join(dataset_info)
# df_stats[feature] = df_stats[feature] - df_stats.query("phase=='Pre'")[feature].groupby("mouse_id").median()
# df_stats = df_stats.query("phase != 'Pre'")

# statistics using R's lme4 and emmeans packages
with (robjects.default_converter + pandas2ri.converter).context():  
    # model with random effects for mouse
    model = lme4.lmer(f"{feature} ~ p * 1 + g + (1 | mouse_id)", data=df_stats.reset_index())

    # model with crossed random effects for mouse and day
    # model = lme4.lmer(f"accuracy ~ p * g + (1 | mouse_id)", data=df_stats.reset_index())

    # perform pairwise comparisons between groups at each phase, here we use bonferroni
    formula = f"pairwise ~ g | p"
    emm = emmeans.emmeans(model, stats.formula(formula), adjust='bonferroni')
    res = base.summary(emm[1])

display(res)


df_stats.to_csv(p_figures / "figure_1_moverate_contra_raw_data_check.csv")

# plot changes in task performance
f, ax = plt.subplots(1, 1, figsize=(4.5*cm, 4.5*cm))
sns.boxplot(
    data = df_stats.reset_index(),
    x = "p",
    y = feature,
    hue = "g",
    hue_order = ["Sham", "Stroke", "Stroke + training"],
    # order = ["Post Early", "Post Late"],
    palette = group_colors,
    ax = ax,
    showfliers = False,
    fill = False,
    legend=False,
    whis=(0,100),
    gap=0.2
)

# add statistical annotations
pairs = []
p_values = []
for i, row in res.iterrows():
    g1, g2 = row['contrast'].split(" - ")
    if g1 == "(Stroke + training)":
        g1 = "Stroke + training"
    if g2 == "(Stroke + training)":
        g2 = "Stroke + training"
    pairs.append(((row['p'], g1), (row['p'], g2)))
    p_values.append(row['p.value'])

annotator = Annotator(ax, pairs, data=df_stats.reset_index(), x="phase", y=feature, hue="stroke_group", hue_order=["Sham", "Stroke", "Stroke + training"])
annotator.configure(test=None, text_format='star', loc='inside', verbose=0, hide_non_significant=True,
                    fontsize=fontsize_small, line_width=1, text_offset=0, line_offset=0, use_fixed_offset=True)
annotator.set_pvalues(p_values)
annotator._pvalue_format.config(pvalue_thresholds=[(1e-3, '***'), (1e-2, '**'), (0.05, '*'), (1, 'ns')])
annotator.annotate()

sns.stripplot(
    data = df_stats.reset_index(),
    x = "phase",
    y = feature,
    hue = "stroke_group",
    palette = group_colors,
    ax = ax,
    dodge = True,
    size = 3,
    alpha = 0.5,
    legend = False,
    hue_order = ["Sham", "Stroke", "Stroke + training"],
)
ax.set_ylabel("support limb\nmovement rate [s$^{-1}$]")
ax.set_xlabel("Phase")
ax.set_xticks([0, 1, 2], labels=["Pre", "Early", "Late"])
#remove top and right spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
# ax.set_xlim(0.5, 2.5)
f.tight_layout()
f.savefig(p_figures / "figure_1_moverate_contra_raw.svg", dpi=300, transparent=True)
plt.show(f)
plt.close(f)
res.to_csv(p_figures / "figure_1_moverate_contra_raw_stats.csv")
df_stats.to_csv(p_figures / "figure_1_moverate_contra_raw_data.csv")

### Supplementary figure 1C - support limb fine motor features

In [ ]:
features_to_test = ["rotation_24_range", "open_alt_24_mean", "bend_24_mean"]
labels = ["Limb rotation range (°)", "Hand aperture (°)", "Finger bend (°)"]

f, ax = plt.subplots(1, 3, figsize=(3 * 3.5*cm, 4.5*cm), sharey=False,
                     gridspec_kw={'wspace': 0.4, 'left': 0.1, 'right': 0.95})
for i in range(3):
    feature_to_test = features_to_test[i]
    ylabel = labels[i]

    df_stats = df_onset_offset_contra.filter([feature_to_test]).join(dataset_info)

    # statistics using R's lme4 and emmeans packages
    with (robjects.default_converter + pandas2ri.converter).context():  
        # model with crossed random effects for mouse and day
        model = lme4.lmer(f"{feature_to_test} ~ 1 + p * g + (1 | mouse_id / d)", data=df_stats.reset_index())

        # estimate marginal means and compute contrasts at each phase between groups
        emm_groups_at_phase = emmeans.emmeans(model, stats.formula("pairwise ~ g | p"), adjust='bonferroni')
        res_groups_at_phase = base.summary(emm_groups_at_phase[1])

    display(res_groups_at_phase)

    # compute medians and iqr for data output
    df_stats_summary = df_stats.reset_index().groupby(["p", "g"])[feature_to_test].agg(['count', 'median',
                                                                                      lambda x: np.percentile(x, 25),
                                                                                      lambda x: np.percentile(x, 75)])
    df_stats_summary.to_csv(p_figures / f"figure_1_contra_{features_to_test[i]}_data.csv")

    # plot changes in feature
    sns.boxplot(
        data = df_stats.reset_index(),
        x = "p",
        y = feature_to_test,
        hue = "g",
        palette = group_colors,
        ax = ax[i],
        showfliers = False,
        fill = False,
        legend=False,
        zorder=2,
        gap=0.2,
    )

    # add statistical annotations
    pairs = []
    p_values = []
    for _, row in res_groups_at_phase.iterrows():
        g1, g2 = row['contrast'].split(" - ")
        if g1 == "(Stroke + training)":
            g1 = "Stroke + training"
        if g2 == "(Stroke + training)":
            g2 = "Stroke + training"
        pairs.append(((row['p'], g1), (row['p'], g2)))
        p_values.append(row['p.value'])

    p_values = np.array(p_values)

    annotator = Annotator(ax[i], pairs, data=df_stats.reset_index(), x="p", y=feature_to_test, hue="g", hue_order=["Sham", "Stroke", "Stroke + training"])
    annotator.configure(test=None, text_format='full', loc='inside', verbose=0, hide_non_significant=True,
                        fontsize=fontsize_small, line_width=1, text_offset=0, line_offset=0, use_fixed_offset=True)
    annotator.set_pvalues(p_values)
    annotator.annotate()


    sns.stripplot(
        data = df_stats.reset_index(),
        x = "p",
        y = feature_to_test,
        hue = "g",
        palette = group_colors,
        ax = ax[i],
        dodge = True,
        size = 2,
        alpha = 0.05,
        jitter=0.15,
        legend = False,
        hue_order = ["Sham", "Stroke", "Stroke + training"],
        zorder=1
    )

    ax[i].set_ylabel(f"{ylabel}")
    ax[i].set_xlabel("")
    ax[i].set_xticks([0, 1, 2], labels=["Pre", "Early", "Late"])
    #remove top and right spines
    ax[i].spines['top'].set_visible(False)
    ax[i].spines['right'].set_visible(False)

f.savefig(p_figures / f"SF1_fine_feature_stats_supportlimb.svg", dpi=600, transparent=True)
plt.show(f)
plt.close(f)
# res.to_csv(Path(p_figures, "features") / f"figure_1_{feature_to_test}_stats.csv")
# df_stats.to_csv(Path(p_figures, "features") / f"figure_1_{feature_to_test}_data.csv")

### Figure 1J - correlations lesion size

In [ ]:
df_stroke_volume = (
    exp.StrokeVolume
    .fetch(format='frame')
    .reset_index()
    .filter(['mouse_id', 'stroke_volume'])
    .dropna()
    .set_index('mouse_id')
)
display(df_stroke_volume)


features_to_test = ["rotation_24_range", "open_alt_24_mean", "bend_24_mean"]
df_onset_offset_ipsi_corr = df_onset_offset_ipsi.copy()
df_onset_offset_ipsi_corr = df_onset_offset_ipsi_corr - df_onset_offset_ipsi_corr.query("day<0").groupby("mouse_id").median()
df_onset_offset_ipsi_corr = df_onset_offset_ipsi_corr.join(dataset_info).groupby(["mouse_id", "p"])[features_to_test].mean()


labels = ["Rotation range (°)", "Hand aperture (°)", "Finger bend (°)"]
# df_summary = df_onset_offset_ipsi_corr

# day 3 vs day 28
from scipy.stats import spearmanr, pearsonr
df_summary = df_onset_offset_ipsi_corr.join(df_stroke_volume).query("p!='Pre'").dropna().join(dataset_info.reset_index().filter(["mouse_id", "stroke_group"]).groupby("mouse_id").first(), on="mouse_id")
df_summary.to_csv(p_figures / f"figure_1_fine_feature_correlation_stroke_volume_data.csv")
display(df_summary)

d3 = df_onset_offset_ipsi_corr.join(df_stroke_volume).query("p=='Post Early'").dropna().join(dataset_info.reset_index().filter(["mouse_id", "stroke_group"]).groupby("mouse_id").first(), on="mouse_id")
d28 = df_onset_offset_ipsi_corr.join(df_stroke_volume).query("p=='Post Late'").dropna().join(dataset_info.reset_index().filter(["mouse_id", "stroke_group"]).groupby("mouse_id").first(), on="mouse_id")
f, ax = plt.subplots(2,3, sharex=True, figsize=(10*cm, 6*cm), gridspec_kw={'hspace': 0.1, 'wspace': 0.3})
for i, feature in enumerate(features_to_test):
    colors_early = [group_colors[g] for g in d3.stroke_group]
    ax[0,i].scatter(d3["stroke_volume"], d3[feature], c=colors_early, alpha=.7, edgecolors='none')
    r, p = spearmanr(d3["stroke_volume"], d3[feature])
    ax[0, i].text(0.05, 0.9, f"$\\rho$={r:.2f}\np={p:.3f}", ha='left', va='top', transform=ax[0,i].transAxes, fontsize=fontsize_small)
    colors_late = [group_colors[g] for g in d28.stroke_group]
    ax[1,i].scatter(d28["stroke_volume"], d28[feature], c=colors_late, alpha=.7, edgecolors='none')
    r, p = spearmanr(d28["stroke_volume"], d28[feature])
    ax[1, i].text(0.05, 0.95, f"$\\rho$={r:.2f}\np={p:.3f}", ha='left', va='top', transform=ax[1,i].transAxes, fontsize=fontsize_small)
    ax[0,i].set_title(labels[i])

ax[1,1].set_xlabel("Stroke volume (mm$^3$)")
ax[0,0].set_ylabel("Early")
ax[1,0].set_ylabel("Late")
f.suptitle("Changes in task limb motor features, relative to Pre")
f.savefig(p_figures / f"figure_1_fine_feature_correlation_stroke_volume_rel.svg", dpi=600, transparent=True)
plt.show()

### figure 1J - lesion size vs performance

In [ ]:
# get performance dataset
dataset_performance_rel = dataset_performance - dataset_performance.query("day<0").groupby("mouse_id").median()
dataset_performance_corr = (dataset_performance_rel.join(dataset_info).groupby(["mouse_id", "p"])[["rewarded_rate", "accuracy"]].mean())
display(dataset_performance_corr)

df_stroke_volume = (
    exp.StrokeVolume
    .fetch(format='frame')
    .reset_index()
    .filter(['mouse_id', 'stroke_volume'])
    .dropna()
    .set_index('mouse_id')
)
display(df_stroke_volume)


features_to_test = ["rewarded_rate", "accuracy"]
# df_onset_offset_ipsi_corr = df_onset_offset_ipsi.copy()
# df_onset_offset_ipsi_corr = df_onset_offset_ipsi_corr - df_onset_offset_ipsi_corr.query("day<0").groupby("mouse_id").median()
# df_onset_offset_ipsi_corr = df_onset_offset_ipsi_corr.join(dataset_info).groupby(["mouse_id", "p"])[features_to_test].mean()

labels = ["Reward rate (s$^{-1}$)", "Accuracy (%)"]

# day 3 vs day 28
from scipy.stats import spearmanr, pearsonr

df_summary = dataset_performance_corr.join(df_stroke_volume).query("p!='Pre'").dropna().join(dataset_info.reset_index().filter(["mouse_id", "stroke_group"]).groupby("mouse_id").first(), on="mouse_id")
df_summary.to_csv(p_figures / f"figure_1_performance_correlation_stroke_volume_data.csv")
display(df_summary)

d3 = dataset_performance_corr.join(df_stroke_volume).join(dataset_info.reset_index().filter(["mouse_id", "p", "stroke_group"]).groupby(["mouse_id", "p"]).first()).query("p=='Post Early'").dropna()
d28 = dataset_performance_corr.join(df_stroke_volume).join(dataset_info.reset_index().filter(["mouse_id", "p", "stroke_group"]).groupby(["mouse_id", "p"]).first()).query("p=='Post Late'").dropna()

f, ax = plt.subplots(2,2, sharex=True, figsize=(7*cm, 6*cm), gridspec_kw={'hspace': 0.1, 'wspace': 0.3})
for i, feature in enumerate(features_to_test):
    colors_early = [group_colors[g] for g in d3.stroke_group]
    ax[0,i].scatter(d3["stroke_volume"], d3[feature], c=colors_early, alpha=.7, edgecolors='none')
    r, p = spearmanr(d3["stroke_volume"], d3[feature])
    ax[0, i].text(0.05, 0.9, f"$\\rho$={r:.2f}\np={p:.3f}", ha='left', va='top', transform=ax[0,i].transAxes, fontsize=fontsize_small)
    colors_late = [group_colors[g] for g in d28.stroke_group]
    ax[1,i].scatter(d28["stroke_volume"], d28[feature], c=colors_late, alpha=.7, edgecolors='none')
    r, p = spearmanr(d28["stroke_volume"], d28[feature])
    ax[1, i].text(0.05, 0.95, f"$\\rho$={r:.2f}\np={p:.3f}", ha='left', va='top', transform=ax[1,i].transAxes, fontsize=fontsize_small)
    ax[0,i].set_title(labels[i])

ax[1,1].set_xlabel("Stroke volume (mm$^3$)")
ax[0,0].set_ylabel("Early")
ax[1,0].set_ylabel("Late")
f.suptitle("Changes in task outcomes, relative to Pre")
f.savefig(p_figures / f"figure_1_performance_correlation_stroke_volume_rel.svg", dpi=600, transparent=True)
plt.show()

### Figure 1J - correlations with motor features (rel to pre)

In [ ]:
features_to_test = ["rotation_24_range", "open_alt_24_mean", "bend_24_mean"]
df_onset_offset_ipsi_corr = df_onset_offset_ipsi.copy()
df_onset_offset_ipsi_corr = df_onset_offset_ipsi_corr - df_onset_offset_ipsi_corr.query("day<0").groupby("mouse_id").median()
df_onset_offset_ipsi_corr = df_onset_offset_ipsi_corr.join(dataset_info).groupby(["mouse_id", "day"])[features_to_test].mean()
labels = ["Rotation range (°)", "Hand aperture (°)", "Finger bend (°)"]
df_performance_corr = dataset_performance.query("performance > 5").copy()
df_performance_corr = df_performance_corr.join(dataset_info).groupby(["mouse_id", "day"])[["rewarded_rate", "performance", "accuracy"]].mean()
perf_to_test = "performance"
perf_label = "task performance (%)"

# day 3 vs day 28
from scipy.stats import spearmanr, pearsonr
d3 = df_onset_offset_ipsi_corr.join(df_performance_corr).join(dataset_info).query("p=='Post Early'").dropna()
d28 = df_onset_offset_ipsi_corr.join(df_performance_corr).join(dataset_info).query("p=='Post Late'").dropna()
# display(d3)
# display(d28)
# output data for manuscript submission
df_summary = df_onset_offset_ipsi_corr.join(df_performance_corr).join(dataset_info).query("p!='Pre'").dropna()
display(df_summary.groupby("stroke_group").count())
df_summary.to_csv(p_figures / f"figure_1_fine_feature_performance_correlation_data_{perf_to_test}.csv")
# display(df_onset_offset_ipsi_corr.join(df_performance_corr).join(dataset_info).query("p!='Pre'").dropna())
f, ax = plt.subplots(2,3, sharex=True, figsize=(10*cm, 6*cm), gridspec_kw={'hspace': 0.1, 'wspace': 0.3}, sharey='col')
for i, feature in enumerate(features_to_test):
    colors_early = [group_colors[g] for g in d3['g']]
    ax[0,i].scatter(d3[perf_to_test], d3[feature], c=colors_early, alpha=.7, edgecolors='none')
    r, p = spearmanr(d3[perf_to_test], d3[feature])
    ax[0, i].text(0.05, 0.95, f"$\\rho$={r:.2f}\np={p:.3f}", ha='left', va='top', transform=ax[0,i].transAxes, fontsize=fontsize_small)
    colors_late = [group_colors[g] for g in d28['g']]
    ax[1,i].scatter(d28[perf_to_test], d28[feature], c=colors_late, alpha=.7, edgecolors='none')
    r, p = spearmanr(d28[perf_to_test], d28[feature])
    ax[1, i].text(0.05, 0.95, f"$\\rho$={r:.2f}\np={p:.3f}", ha='left', va='top', transform=ax[1,i].transAxes, fontsize=fontsize_small)
    ax[0,i].set_title(labels[i])

ax[1,1].set_xlabel(f"{perf_label}")
ax[0,0].set_ylabel("Early")
ax[1,0].set_ylabel("Late")
f.suptitle("Changes in motor features, relative to Pre")
f.savefig(p_figures / f"figure_1_fine_feature_correlation_rel_{perf_to_test}.svg", dpi=600, transparent=True)
plt.show()

## Supplementary figure 1A-B

In [ ]:
mouse_id = 55
keys = (pt.MovementSegmentation.Hand.proj() * pt.PawRecording.Hand * exp.ExperimentalPhase & dict(mouse_id=mouse_id, side='ipsi', phase='Expert')).fetch('KEY')
print(keys)
df_epoch_features = (
    (pt.MovementSegmentation.Hand
    & keys
    )
    .epoch_features(labels_to_include = ["open_alt_24", "bend_24", "rotation_24", "velocity", "distance"])
    .join((pt.EpochClassification.Epoch.proj("epoch_class") & keys).fetch(format='frame'))
    .reset_index()
    .drop(columns=["username", "session_num", "pt_seg_id", "hand"])
    .set_index(["mouse_id", "day", "epoch_id", "epoch_class"])
    .query("epoch_class != 'excluded'")
)
df_epoch_features

X = df_epoch_features.to_numpy()
from sklearn.preprocessing import StandardScaler
X_scaled = StandardScaler().fit_transform(X)
from umap.umap_ import UMAP
reducer = UMAP(n_neighbors=15, min_dist=0.1, n_components=2)
embedding = reducer.fit_transform(X_scaled)

color_palette = dict(rewarded="tab:blue", miss="tab:orange", other="tab:green")
colors = [color_palette[c] for c in df_epoch_features.reset_index().epoch_class]
plt.figure(figsize=(4*cm, 4*cm))
plt.scatter(embedding[:, 0], embedding[:, 1], s=2, alpha=0.5, c=colors, edgecolors='none')
plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")

# export data
df_umap = pd.DataFrame(embedding, columns=["UMAP1", "UMAP2"], index=df_epoch_features.index)
df_umap.reset_index().to_csv(p_figures / f"SF1_expert_umap_mouse_{mouse_id}_data.csv")
# df_umap["epoch_class"] = df_epoch_features.reset_index().epoch_class.values
display(df_umap)

# set aspect to equal
plt.gca().set_aspect('equal', 'box')
plt.savefig(p_figures / f"SF1_expert_umap_mouse_{mouse_id}.svg", dpi=600, transparent=True)
plt.show()

In [ ]:
keys = (
    pt.MovementSegmentation.Hand.proj("side")
    * exp.DaysFromStrokeNorm
    * exp.ExperimentalPhase
    * exp.StrokeGroup.proj(stroke_group='group')
    & "phase = 'Expert'"
    & "stroke_group != 'Learning'"
    & [f"days_from_stroke_norm = {d}" for d in [-3, -2, -1, 3, 7, 14, 21, 28]]
    & "mouse_id > 40"
    & "side = 'ipsi'"
).fetch("KEY")
df_epoch_features = (
    (pt.MovementSegmentation.Hand
    & keys
    )
    .epoch_features(labels_to_include = ["open_alt_24", "bend_24", "rotation_24", "velocity", "distance"])
    .join((pt.EpochClassification.Epoch.proj("epoch_class") & keys).fetch(format='frame'))
    .reset_index()
    .drop(columns=["username", "session_num", "pt_seg_id", "hand"])
    .set_index(["mouse_id", "day", "epoch_id", "epoch_class"])
    .query("epoch_class != 'excluded'")
)
df_epoch_features["rotation_24_range"] = df_epoch_features["rotation_24_max"] - df_epoch_features["rotation_24_min"]

In [ ]:
# get counts
df_epoch_features.groupby("epoch_class").count()

# get median, 25th and 75th percentiles by epoch class
df_summary = df_epoch_features.groupby("epoch_class").agg(["count", "median", lambda x: x.quantile(0.25), lambda x: x.quantile(0.75)])
df_summary.to_csv(p_figures / f"SF1_expert_umap_epoch_feature_summary.csv")
df_summary

In [ ]:
f, ax = plt.subplots(1, 3, figsize=(7*cm, 4.5*cm))
sns.boxplot(
    data = df_epoch_features.reset_index(),
    hue = "epoch_class",
    y = "rotation_24_range",
    palette = dict(rewarded="tab:blue", miss="tab:orange", other="tab:green"),
    ax = ax[0],
    showfliers = False,
    fill = False,
    legend=False,
    dodge=True,
    width=0.5,
    gap = 0.2
)

sns.boxplot(
    data = df_epoch_features.reset_index(),
    hue = "epoch_class",
    y = "open_alt_24_mean",
    palette = dict(rewarded="tab:blue", miss="tab:orange", other="tab:green"),
    ax = ax[1],
    showfliers = False,
    fill = False,
    legend=False,
    dodge=True,
    width=0.5,
    gap = 0.2
)

sns.boxplot(
    data = df_epoch_features.reset_index(),
    hue = "epoch_class",
    y = "bend_24_mean",
    palette = dict(rewarded="tab:blue", miss="tab:orange", other="tab:green"),
    ax = ax[2],
    showfliers = False,
    fill = False,
    legend=False,
    dodge=True,
    width=0.5,
    gap = 0.2
)

# sns.stripplot(
#     data = df_epoch_features.reset_index(),
#     hue = "epoch_class",
#     y = feature,
#     palette = dict(rewarded="tab:blue", miss="tab:orange", other="tab:green"),
#     ax = ax,
#     dodge = True,
#     size = 3,
#     alpha = 0.05,
#     legend = False,
# )
# f.tight_layout()
f.savefig(p_figures / f"SF1_expert_epoch_features.svg", dpi=600, transparent=True)
plt.show()